In [1]:
import pandas as pd
import os

DIR1 = '../data/raw/01'
DIR2 = '../data/raw/02'
INTERIM = '../data/interim'
os.makedirs(INTERIM, exist_ok=True)

In [2]:
def load_and_merge(filename):
    dfs = []
    for d in [DIR1, DIR2]:
        path = f'{d}/{filename}'
        if os.path.exists(path):
            df = pd.read_csv(path)
            df['ExportPeriod'] = '1구간' if d == DIR1 else '2구간'
            dfs.append(df)
    if not dfs:
        print(f'{filename} 없음')
        return None
    merged = pd.concat(dfs, ignore_index=True)
    
    date_col = next((c for c in ['ActivityDate','SleepDay','Date','ActivityHour','Time'] if c in merged.columns), None)
    if date_col:
        before = len(merged)
        merged = merged.drop_duplicates(subset=['Id', date_col])
        if before != len(merged):
            print(f'   ↳ {filename}: 중복 {before-len(merged)}건 제거')
    merged.to_csv(f'{INTERIM}/{filename}', index=False)
    return merged

In [3]:
files_to_merge = [
    'dailyActivity_merged.csv',
    'sleepDay_merged.csv',
    'weightLogInfo_merged.csv',
    'heartrate_seconds_merged.csv',
    'hourlySteps_merged.csv',
    'hourlyCalories_merged.csv',
    'hourlyIntensities_merged.csv',
]

for f in files_to_merge:
    df = load_and_merge(f)
    if df is not None:
        print(f'{f:<32} 병합행수={len(df):<8} 고유Id={df["Id"].nunique() if "Id" in df.columns else "-"}')

   ↳ dailyActivity_merged.csv: 중복 24건 제거
dailyActivity_merged.csv         병합행수=1373     고유Id=35
   ↳ sleepDay_merged.csv: 중복 3건 제거
sleepDay_merged.csv              병합행수=410      고유Id=24
   ↳ weightLogInfo_merged.csv: 중복 2건 제거
weightLogInfo_merged.csv         병합행수=98       고유Id=13
   ↳ heartrate_seconds_merged.csv: 중복 23424건 제거
heartrate_seconds_merged.csv     병합행수=3614915  고유Id=15
   ↳ hourlySteps_merged.csv: 중복 175건 제거
hourlySteps_merged.csv           병합행수=46008    고유Id=35
   ↳ hourlyCalories_merged.csv: 중복 175건 제거
hourlyCalories_merged.csv        병합행수=46008    고유Id=35
   ↳ hourlyIntensities_merged.csv: 중복 175건 제거
hourlyIntensities_merged.csv     병합행수=46008    고유Id=35


In [4]:
ids_1 = set(pd.read_csv(f'{DIR1}/dailyActivity_merged.csv')['Id'])
ids_2 = set(pd.read_csv(f'{DIR2}/dailyActivity_merged.csv')['Id'])

print('1구간 사용자 수:', len(ids_1))
print('2구간 사용자 수:', len(ids_2))
print('두 구간 모두 있는 사용자:', len(ids_1 & ids_2))
print('1구간에만 있는 사용자:', len(ids_1 - ids_2))
print('2구간에만 있는 사용자:', len(ids_2 - ids_1))

1구간 사용자 수: 35
2구간 사용자 수: 33
두 구간 모두 있는 사용자: 33
1구간에만 있는 사용자: 2
2구간에만 있는 사용자: 0


In [5]:
da = pd.read_csv(f'{INTERIM}/dailyActivity_merged.csv')
da['ActivityDate'] = pd.to_datetime(da['ActivityDate'], format='%m/%d/%Y')
print('전체 사용자 수:', da['Id'].nunique())
print('날짜 범위:', da['ActivityDate'].min(), '~', da['ActivityDate'].max())
print('Id+Date 중복행:', da.duplicated(subset=['Id','ActivityDate']).sum())

전체 사용자 수: 35
날짜 범위: 2016-03-12 00:00:00 ~ 2016-05-12 00:00:00
Id+Date 중복행: 0
